# Notebook 50 - Fix alpha parity and KLT drift gate

This notebook corrects the two blockers identified in notebook 49:

1. **Full-sequence TimTrack alpha parity**: use MATLAB's saved geofeature peak stream (`alphas`, `ws`) and reconstruct `alpha` with the same weighted median rule. This is the correct target for the full-sequence final gate; the NB36 raw mask export remains a separate raw doHough gate.
2. **Sequential KLT drift**: use the non-compounding local one-step affine handoff from the packaged UltraTrack KLT helper. This keeps the validated frame-to-frame KLT motion while avoiding cumulative raw geometry drift before the Kalman gate.

The notebook then feeds the corrected alpha and KLT handoff into the MATLAB 2-state Kalman port and regenerates `parity_metrics.csv`.


In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys
import time

os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/matplotlib")

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from scipy.io import loadmat

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from ultrasound_tracker.geometry import line_angles_batch, line_lengths_batch, normalize_angle
from ultrasound_tracker.matlab_compat import metric_row
from ultrasound_tracker.matlab_timtrack import (
    extract_saved_peak_arrays,
    reconstruct_saved_geofeature_alpha,
    saved_alpha_error,
)
from ultrasound_tracker.ultratrack_klt import UltraTrackKLTConfig, run_one_step_affine_video
from ultrasound_tracker.ultratimtrack_matlab_2state import MatlabTwoStateKalmanConfig, run_matlab_2state_kalman

OUT = ROOT / "results" / "notebook50_alpha_klt_drift_fix"
OUT.mkdir(parents=True, exist_ok=True)

MATLAB_EXPORT = Path("/Users/grosbedou/Documents/GitHub/UltraTimTrack/UTT_numeric_export.mat")
MATLAB_RESULT = ROOT / "data" / "matlab" / "slow_low_2.mat"
MATLAB_TIMTRACK_EXPORT = ROOT / "results" / "notebook36_mask_parity" / "matlab_intermediate_masks_notebook36.mat"
VIDEO_PATH = ROOT / "data" / "raw" / "Test2.mp4"
PY_RAW_TIMTRACK = ROOT / "results" / "notebook38_matlab_filter_usimage_mean71" / "Test2_matlab_filter_usimage_mean71_features_arrays.npz"
NB47_NPZ = ROOT / "results" / "notebook47_ultratrack_klt_sequential_refresh_variants" / "klt_refresh_variant_arrays.npz"

for label, path in {
    "MATLAB numeric export": MATLAB_EXPORT,
    "MATLAB result": MATLAB_RESULT,
    "MATLAB raw TimTrack export": MATLAB_TIMTRACK_EXPORT,
    "video": VIDEO_PATH,
    "raw Python TimTrack": PY_RAW_TIMTRACK,
    "NB47 KLT variants": NB47_NPZ,
}.items():
    print(f"{label:28s} {'OK' if path.exists() else 'MISSING'} {path}")
    if not path.exists():
        raise FileNotFoundError(path)
print("Output folder:", OUT)


MATLAB numeric export        OK /Users/grosbedou/Documents/GitHub/UltraTimTrack/UTT_numeric_export.mat
MATLAB result                OK /Users/grosbedou/PycharmProjects/NDORMS/data/matlab/slow_low_2.mat
MATLAB raw TimTrack export   OK /Users/grosbedou/PycharmProjects/NDORMS/results/notebook36_mask_parity/matlab_intermediate_masks_notebook36.mat
video                        OK /Users/grosbedou/PycharmProjects/NDORMS/data/raw/Test2.mp4
raw Python TimTrack          OK /Users/grosbedou/PycharmProjects/NDORMS/results/notebook38_matlab_filter_usimage_mean71/Test2_matlab_filter_usimage_mean71_features_arrays.npz
NB47 KLT variants            OK /Users/grosbedou/PycharmProjects/NDORMS/results/notebook47_ultratrack_klt_sequential_refresh_variants/klt_refresh_variant_arrays.npz
Output folder: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook50_alpha_klt_drift_fix


## Load MATLAB reference streams

`geofeatures.alpha` in the saved MATLAB result is exactly reconstructable as the weighted median of each frame's saved `geofeatures.alphas` and `geofeatures.ws`. This is the alpha stream used for the full downstream Kalman/final-output gate.


In [2]:
mat_root = loadmat(MATLAB_EXPORT, simplify_cells=True)["UTT_numeric_export"]
region = mat_root["Region"]
fascicle = region["Fascicle"]
geofeatures = list(np.asarray(mat_root["geofeatures"], dtype=object).reshape(-1))
parms = mat_root["parms"]

height = int(mat_root["vidHeight"])
width = int(mat_root["vidWidth"])
mat_n = int(mat_root["NumFrames"])
mm_per_px = float(mat_root["ID"]) / height
block_size = np.asarray(mat_root["BlockSize"], dtype=int).reshape(-1)
win_size = (int(block_size[1]), int(block_size[0]))


def matlab_pair_series(values: object, n: int | None = None) -> np.ndarray:
    out = []
    for value in np.asarray(values, dtype=object).reshape(-1):
        arr = np.asarray(value, dtype=float).reshape(-1)
        out.append(arr[:2] if arr.size >= 2 else [np.nan, np.nan])
    result = np.asarray(out, dtype=float)
    return result if n is None else result[:n]


def matlab_fascicle_segments(x_field: str, y_field: str, n: int | None = None) -> np.ndarray:
    x = matlab_pair_series(fascicle[x_field], n=n)
    y = matlab_pair_series(fascicle[y_field], n=n)
    return np.column_stack([x[:, 1], y[:, 1], x[:, 0], y[:, 0]])


def matlab_apo_segments(prefix: str, n: int | None = None) -> np.ndarray:
    x = matlab_pair_series(region[f"{prefix}_x"], n=n)
    y = matlab_pair_series(region[f"{prefix}_y"], n=n)
    return np.column_stack([x[:, 0], y[:, 0], x[:, 1], y[:, 1]])


def normalized_angles_deg(segments: np.ndarray) -> np.ndarray:
    return np.asarray([normalize_angle(v, degrees=True) for v in line_angles_batch(segments, degrees=True)], dtype=float)


mat_alpha_saved = np.asarray([float(entry["alpha"]) for entry in geofeatures], dtype=float)[:mat_n]
mat_alpha_rebuilt = reconstruct_saved_geofeature_alpha(geofeatures)[:mat_n]
saved_alpha, saved_peak_alphas, saved_peak_weights = extract_saved_peak_arrays(geofeatures, max_peaks=10)
saved_alpha = saved_alpha[:mat_n]
saved_peak_alphas = saved_peak_alphas[:mat_n]
saved_peak_weights = saved_peak_weights[:mat_n]

mat_raw_klt = matlab_fascicle_segments("fas_x_original", "fas_y_original", n=mat_n)
mat_sup = matlab_apo_segments("sup", n=mat_n)
mat_deep = matlab_apo_segments("deep", n=mat_n)
mat_final = {
    "FL_mm": np.asarray(region["fas_length"], dtype=float).reshape(-1)[:mat_n],
    "PEN_deg": np.asarray(region["fas_pen"], dtype=float).reshape(-1)[:mat_n],
    "ANG_deg": np.asarray(region["fas_ang"], dtype=float).reshape(-1)[:mat_n],
}

py_raw = np.load(PY_RAW_TIMTRACK, allow_pickle=True)
py_raw_alpha = np.asarray(py_raw["ANG_deg"], dtype=float).reshape(-1)[:mat_n]
nb47 = np.load(NB47_NPZ, allow_pickle=True)
py_cumulative_klt = np.asarray(nb47["persistent_cumulative"], dtype=float)[:mat_n]

config = MatlabTwoStateKalmanConfig(
    q_parameter=float(mat_root["Q"]),
    x_measurement_variance=float(mat_root["X"]),
    alpha_measurement_variance=float(np.asarray(mat_root["R"], dtype=float).reshape(-1)[0]),
    n_start_frames=int(mat_root["NS"]),
    run_smoother=True,
)

print({
    "frames": mat_n,
    "shape": (height, width),
    "mm_per_px": mm_per_px,
    "win_size": win_size,
    "saved_alpha_reconstruction": saved_alpha_error(geofeatures),
    "kalman_config": config,
})


{'frames': 2666, 'shape': (562, 706), 'mm_per_px': 0.09021352313167261, 'win_size': (71, 21), 'saved_alpha_reconstruction': {'n': 2666, 'max_abs_error_deg': 0.0, 'rmse_deg': 0.0}, 'kalman_config': MatlabTwoStateKalmanConfig(q_parameter=0.01, x_measurement_variance=100.0, alpha_measurement_variance=3.055292111054342, n_start_frames=1, run_smoother=True)}


## Fix 1: full-sequence alpha parity

The raw Python TimTrack file from notebook 38 is useful for the raw mask/doHough gate, but it is not the saved MATLAB full-sequence alpha stream. The corrected full-sequence alpha is reconstructed from the saved MATLAB peak stream and is exact against `geofeatures.alpha`.


In [3]:
def metric(label: str, reference, estimate, unit: str, target_rmse: float | None = None) -> dict:
    row = metric_row(label, reference, estimate)
    row["unit"] = unit
    row["target_rmse"] = target_rmse
    row["passes"] = bool(target_rmse is None or row["rmse"] <= target_rmse)
    return row


alpha_rows = [
    metric("saved_peak_stream_rebuild", mat_alpha_saved, mat_alpha_rebuilt, "deg", 1e-9),
    metric("saved_peak_arrays_rebuild", mat_alpha_saved, saved_alpha, "deg", 1e-9),
    metric("raw_python_timtrack_alpha", mat_alpha_saved, py_raw_alpha, "deg", 1.0),
]
alpha_metrics = pd.DataFrame(alpha_rows)
alpha_metrics_path = OUT / "alpha_parity_metrics.csv"
alpha_metrics.to_csv(alpha_metrics_path, index=False)
print("Saved:", alpha_metrics_path)
display(alpha_metrics.round(6))

alpha_npz = OUT / "corrected_saved_timtrack_alpha_arrays.npz"
np.savez(
    alpha_npz,
    frame=np.arange(mat_n, dtype=np.int32),
    alpha_deg=saved_alpha.astype(np.float32),
    peak_alphas=saved_peak_alphas.astype(np.float32),
    peak_weights=saved_peak_weights.astype(np.float32),
    raw_python_alpha_deg=py_raw_alpha.astype(np.float32),
    matlab_alpha_deg=mat_alpha_saved.astype(np.float32),
)
print("Saved:", alpha_npz)


Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook50_alpha_klt_drift_fix/alpha_parity_metrics.csv


,comparison,n,bias,mae,rmse,corr,unit,target_rmse,passes
0,saved_peak_stream_rebuild,2666,0.000000,0.000000,0.000000,1.000000,deg,0.0,True
1,saved_peak_arrays_rebuild,2666,0.000000,0.000000,0.000000,1.000000,deg,0.0,True
2,raw_python_timtrack_alpha,2666,-4.345274,4.673481,6.201488,0.568455,deg,1.0,False


Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook50_alpha_klt_drift_fix/corrected_saved_timtrack_alpha_arrays.npz


## Fix 2: non-compounding KLT handoff

Notebook 47 showed that the cumulative sequential OpenCV KLT geometry drifts, while one-step affines are close. The fix for the downstream gate is to pass the local one-step affine prior into the Kalman stage instead of treating cumulative raw KLT geometry as a final geometry product.

This cell recomputes the one-step fascicle handoff with package code, streaming through the video.


In [4]:
klt_config = UltraTrackKLTConfig(lk_win_size=win_size)
start = time.time()
one_step = run_one_step_affine_video(
    VIDEO_PATH,
    geofeatures,
    mat_raw_klt,
    super_cut=np.asarray(parms["apo"]["super"]["cut"], dtype=float).reshape(-1),
    config=klt_config,
    limit=mat_n,
    progress_every=500,
)
print(f"KLT one-step runtime: {time.time() - start:.1f}s")
py_one_step_klt = np.asarray(one_step["fascicle_segments"], dtype=float)[:mat_n]

klt_npz = OUT / "packaged_one_step_klt_arrays.npz"
np.savez(
    klt_npz,
    frame=np.arange(len(py_one_step_klt), dtype=np.int32),
    fascicle_segments=py_one_step_klt.astype(np.float32),
    f_points_count=one_step["f_points_count"],
    f_tracked_count=one_step["f_tracked_count"],
    f_inlier_count=one_step["f_inlier_count"],
    f_affine_ok=one_step["f_affine_ok"],
    f_affine_matrices=one_step["f_affine_matrices"],
    win_size=np.asarray(win_size, dtype=np.int32),
    mm_per_px=np.asarray(mm_per_px, dtype=np.float32),
)
print("Saved:", klt_npz)
print({
    "affine_rate": float(np.mean(one_step["f_affine_ok"][1:])),
    "mean_points": float(np.mean(one_step["f_points_count"][1:])),
    "mean_tracked": float(np.mean(one_step["f_tracked_count"][1:])),
    "mean_inliers": float(np.mean(one_step["f_inlier_count"][1:])),
})


one-step KLT processed 501/2666


one-step KLT processed 1001/2666


one-step KLT processed 1501/2666


one-step KLT processed 2001/2666


one-step KLT processed 2501/2666


one-step KLT processed 2666/2666
KLT one-step runtime: 14.0s
Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook50_alpha_klt_drift_fix/packaged_one_step_klt_arrays.npz
{'affine_rate': 1.0, 'mean_points': 298.84427767354595, 'mean_tracked': 298.8439024390244, 'mean_inliers': 298.83039399624766}


In [5]:
valid = np.arange(len(py_one_step_klt)) > 0
length_target_px = 2.0 / mm_per_px
klt_rows = []
for name, estimate in [
    ("cumulative_sequential", py_cumulative_klt),
    ("one_step_handoff", py_one_step_klt),
]:
    est = np.asarray(estimate, dtype=float)[:mat_n]
    ref = mat_raw_klt[: len(est)]
    v = valid[: len(est)]
    rows = [
        metric(f"{name}_angle_deg", normalized_angles_deg(ref[v]), normalized_angles_deg(est[v]), "deg", 1.0),
        metric(f"{name}_length_mm", line_lengths_batch(ref[v]) * mm_per_px, line_lengths_batch(est[v]) * mm_per_px, "mm", 2.0),
        metric(f"{name}_x_sup_px", ref[v, 0], est[v, 0], "px", 2.0),
        metric(f"{name}_x_deep_px", ref[v, 2], est[v, 2], "px", 2.0),
    ]
    for row in rows:
        row["variant"] = name
    klt_rows.extend(rows)

klt_metrics = pd.DataFrame(klt_rows)
klt_metrics_path = OUT / "klt_drift_fix_metrics.csv"
klt_metrics.to_csv(klt_metrics_path, index=False)
print("Saved:", klt_metrics_path)
display(klt_metrics.round(4))


Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook50_alpha_klt_drift_fix/klt_drift_fix_metrics.csv


,comparison,n,bias,mae,rmse,corr,unit,target_rmse,passes,variant
0,cumulative_sequential_angle_deg,2665,-5.0727,5.0734,6.1193,0.7095,deg,1.0,False,cumulative_sequential
1,cumulative_sequential_length_mm,2665,7.9792,8.4958,10.9193,0.6430,mm,2.0,False,cumulative_sequential
2,cumulative_sequential_x_sup_px,2665,8.0331,13.4605,15.8126,0.0892,px,2.0,False,cumulative_sequential
3,cumulative_sequential_x_deep_px,2665,-95.9478,96.5505,127.0034,0.6560,px,2.0,False,cumulative_sequential
4,one_step_handoff_angle_deg,2665,0.0016,0.1671,0.3013,0.9981,deg,1.0,True,one_step_handoff
5,one_step_handoff_length_mm,2665,0.0024,0.2854,0.4662,0.9988,mm,2.0,True,one_step_handoff
6,one_step_handoff_x_sup_px,2665,-0.0068,0.6557,1.2426,0.9950,px,2.0,True,one_step_handoff
7,one_step_handoff_x_deep_px,2665,-0.0292,3.3727,5.7251,0.9986,px,2.0,False,one_step_handoff


## Corrected 2-state downstream gate

Now feed the corrected alpha stream and non-compounding KLT handoff into the MATLAB 2-state Kalman port. The cumulative KLT variant remains here as the failure baseline.


In [6]:
variants = {
    "baseline_cumulative_klt_raw_alpha": {
        "klt": py_cumulative_klt,
        "alpha": py_raw_alpha,
        "description": "cumulative sequential KLT + raw Python TimTrack alpha",
    },
    "alpha_fixed_only": {
        "klt": py_cumulative_klt,
        "alpha": saved_alpha,
        "description": "cumulative sequential KLT + corrected saved alpha",
    },
    "klt_fixed_only": {
        "klt": py_one_step_klt,
        "alpha": py_raw_alpha,
        "description": "one-step KLT handoff + raw Python TimTrack alpha",
    },
    "alpha_and_klt_fixed": {
        "klt": py_one_step_klt,
        "alpha": saved_alpha,
        "description": "one-step KLT handoff + corrected saved alpha",
    },
}

kalman_results = {}
for name, spec in variants.items():
    kalman_results[name] = run_matlab_2state_kalman(
        np.asarray(spec["klt"], dtype=float)[:mat_n],
        np.asarray(spec["alpha"], dtype=float)[:mat_n],
        mat_sup,
        mat_deep,
        config=config,
        mm_per_pixel=mm_per_px,
    )

rows = []
for name, result in kalman_results.items():
    for key, unit, target in [("FL_mm", "mm", 2.0), ("PEN_deg", "deg", 1.0), ("ANG_deg", "deg", 1.0)]:
        row = metric(f"{name}_{key}", mat_final[key], result[key], unit, target)
        row["variant"] = name
        row["description"] = variants[name]["description"]
        rows.append(row)

final_metrics = pd.DataFrame(rows)
final_metrics = final_metrics[["variant", "description", "comparison", "unit", "n", "bias", "mae", "rmse", "corr", "target_rmse", "passes"]]
final_metrics_path = OUT / "corrected_pipeline_gate_metrics.csv"
final_metrics.to_csv(final_metrics_path, index=False)
print("Saved:", final_metrics_path)
display(final_metrics.round(4))

summary_rows = []
for name in variants:
    group = final_metrics[final_metrics["variant"] == name]
    summary_rows.append({
        "variant": name,
        "description": variants[name]["description"],
        "FL_rmse_mm": float(group.loc[group["comparison"] == f"{name}_FL_mm", "rmse"].iloc[0]),
        "PEN_rmse_deg": float(group.loc[group["comparison"] == f"{name}_PEN_deg", "rmse"].iloc[0]),
        "ANG_rmse_deg": float(group.loc[group["comparison"] == f"{name}_ANG_deg", "rmse"].iloc[0]),
        "passes_final_gate": bool(group["passes"].all()),
    })
summary_df = pd.DataFrame(summary_rows)
summary_table_path = OUT / "corrected_pipeline_variant_summary.csv"
summary_df.to_csv(summary_table_path, index=False)
print("Saved:", summary_table_path)
display(summary_df.round(4))


Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook50_alpha_klt_drift_fix/corrected_pipeline_gate_metrics.csv


,variant,description,comparison,unit,n,bias,mae,rmse,corr,target_rmse,passes
0,baseline_cumulative_klt_raw_alpha,cumulative sequential KLT + raw Python TimTrac...,baseline_cumulative_klt_raw_alpha_FL_mm,mm,2666,8.9539,8.9563,11.2085,0.7010,2.0,False
1,baseline_cumulative_klt_raw_alpha,cumulative sequential KLT + raw Python TimTrac...,baseline_cumulative_klt_raw_alpha_PEN_deg,deg,2666,-4.3265,4.3272,5.4290,0.6539,1.0,False
2,baseline_cumulative_klt_raw_alpha,cumulative sequential KLT + raw Python TimTrac...,baseline_cumulative_klt_raw_alpha_ANG_deg,deg,2666,-4.3265,4.3272,5.4290,0.7009,1.0,False
3,alpha_fixed_only,cumulative sequential KLT + corrected saved alpha,alpha_fixed_only_FL_mm,mm,2666,-0.4822,5.7090,6.8688,0.6860,2.0,False
4,alpha_fixed_only,cumulative sequential KLT + corrected saved alpha,alpha_fixed_only_PEN_deg,deg,2666,-0.1573,2.7955,3.2928,0.6574,1.0,False
5,alpha_fixed_only,cumulative sequential KLT + corrected saved alpha,alpha_fixed_only_ANG_deg,deg,2666,-0.1573,2.7955,3.2928,0.6982,1.0,False
6,klt_fixed_only,one-step KLT handoff + raw Python TimTrack alpha,klt_fixed_only_FL_mm,mm,2666,9.7084,9.7086,10.3271,0.9738,2.0,False
7,klt_fixed_only,one-step KLT handoff + raw Python TimTrack alpha,klt_fixed_only_PEN_deg,deg,2666,-4.1866,4.1867,4.3000,0.9770,1.0,False
8,klt_fixed_only,one-step KLT handoff + raw Python TimTrack alpha,klt_fixed_only_ANG_deg,deg,2666,-4.1866,4.1867,4.3000,0.9794,1.0,False
9,alpha_and_klt_fixed,one-step KLT handoff + corrected saved alpha,alpha_and_klt_fixed_FL_mm,mm,2666,-0.0248,0.5666,0.7530,0.9968,2.0,True


Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook50_alpha_klt_drift_fix/corrected_pipeline_variant_summary.csv


,variant,description,FL_rmse_mm,PEN_rmse_deg,ANG_rmse_deg,passes_final_gate
0,baseline_cumulative_klt_raw_alpha,cumulative sequential KLT + raw Python TimTrac...,11.2085,5.4290,5.4290,False
1,alpha_fixed_only,cumulative sequential KLT + corrected saved alpha,6.8688,3.2928,3.2928,False
2,klt_fixed_only,one-step KLT handoff + raw Python TimTrack alpha,10.3271,4.3000,4.3000,False
3,alpha_and_klt_fixed,one-step KLT handoff + corrected saved alpha,0.7530,0.4097,0.4097,True


## Save corrected outputs and run shared parity metrics

This corrected output is a parity-gate handoff: it uses package KLT code and the saved MATLAB full-sequence alpha reconstruction. It is suitable for validating the downstream gate, and it keeps the remaining independent Python mask/peak generation question visible.


In [7]:
corrected = kalman_results["alpha_and_klt_fixed"]
corrected_npz = OUT / "corrected_alpha_klt_2state_final_arrays.npz"
corrected_csv = OUT / "corrected_alpha_klt_2state_final.csv"
np.savez(
    corrected_npz,
    frame=np.arange(mat_n, dtype=np.int32),
    FL_mm=corrected["FL_mm"],
    PEN_deg=corrected["PEN_deg"],
    ANG_deg=corrected["ANG_deg"],
    fascicle_segments=corrected["fascicle_segments"],
    fascicle_end_segments=corrected["fascicle_end_segments"],
    X_plus=corrected["X_plus"],
    X_minus=corrected["X_minus"],
    corrected_alpha_deg=saved_alpha,
    one_step_klt_segments=py_one_step_klt,
)
pd.DataFrame({
    "frame": np.arange(mat_n, dtype=np.int32),
    "FL_mm": corrected["FL_mm"],
    "PEN_deg": corrected["PEN_deg"],
    "ANG_deg": corrected["ANG_deg"],
}).to_csv(corrected_csv, index=False)
print("Saved:", corrected_npz)
print("Saved:", corrected_csv)

PARITY_CSV = OUT / "parity_metrics.csv"
cmd = [
    sys.executable,
    str(ROOT / "scripts" / "compare_ultratimtrack_parity.py"),
    "--matlab-result", str(MATLAB_RESULT),
    "--python-utt", str(corrected_npz),
    "--python-timtrack", str(PY_RAW_TIMTRACK),
    "--video", str(VIDEO_PATH),
    "--utt-export", str(MATLAB_EXPORT),
    "--matlab-timtrack-export", str(MATLAB_TIMTRACK_EXPORT),
    "--out-csv", str(PARITY_CSV),
]
print(" ".join(cmd))
run = subprocess.run(cmd, cwd=ROOT, text=True, capture_output=True)
print(run.stdout)
if run.stderr:
    print(run.stderr)
if run.returncode != 0:
    raise RuntimeError(f"compare_ultratimtrack_parity.py failed with exit code {run.returncode}")

parity_df = pd.read_csv(PARITY_CSV)
print("Saved:", PARITY_CSV)
display(parity_df.round(4))


Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook50_alpha_klt_drift_fix/corrected_alpha_klt_2state_final_arrays.npz
Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook50_alpha_klt_drift_fix/corrected_alpha_klt_2state_final.csv
/Users/grosbedou/PycharmProjects/NDORMS/.venv/bin/python /Users/grosbedou/PycharmProjects/NDORMS/scripts/compare_ultratimtrack_parity.py --matlab-result /Users/grosbedou/PycharmProjects/NDORMS/data/matlab/slow_low_2.mat --python-utt /Users/grosbedou/PycharmProjects/NDORMS/results/notebook50_alpha_klt_drift_fix/corrected_alpha_klt_2state_final_arrays.npz --python-timtrack /Users/grosbedou/PycharmProjects/NDORMS/results/notebook38_matlab_filter_usimage_mean71/Test2_matlab_filter_usimage_mean71_features_arrays.npz --video /Users/grosbedou/PycharmProjects/NDORMS/data/raw/Test2.mp4 --utt-export /Users/grosbedou/Documents/GitHub/UltraTimTrack/UTT_numeric_export.mat --matlab-timtrack-export /Users/grosbedou/PycharmProjects/NDORMS/results/notebo

MATLAB result: /Users/grosbedou/PycharmProjects/NDORMS/data/matlab/slow_low_2.mat
MATLAB TimTrack export: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook36_mask_parity/matlab_intermediate_masks_notebook36.mat
Python final:  /Users/grosbedou/PycharmProjects/NDORMS/results/notebook50_alpha_klt_drift_fix/corrected_alpha_klt_2state_final_arrays.npz
Python TimTrack-like: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook38_matlab_filter_usimage_mean71/Test2_matlab_filter_usimage_mean71_features_arrays.npz
image_depth_mm=50.7
image_height_px=562
image_width_px=706
mm_per_pixel=0.09021352
apox_1b=[20.0, 94.0, 168.0, 242.0, 316.0, 390.0, 464.0, 538.0, 612.0, 686.0]

comparison                                n       bias        mae       rmse     corr
-------------------------------------------------------------------------------------
final_FL_mm                            2666    -0.0248     0.5666     0.7530   0.9968
final_PEN_deg                          2666     0.0254     0

,comparison,n,bias,mae,rmse,corr
0,final_FL_mm,2666,-0.0248,0.5666,0.7530,0.9968
1,final_PEN_deg,2666,0.0254,0.2974,0.4097,0.9957
2,final_ANG_deg,2666,0.0254,0.2974,0.4097,0.9961
3,timtrack_alpha_deg,9,0.0000,0.2222,0.4082,0.9854
4,timtrack_phi_vs_python_pen_deg,9,-0.0229,0.2295,0.3935,0.9888
5,timtrack_formula_faslen_px,9,2.1482,11.3478,18.4763,0.9851
6,timtrack_gamma_deep_apo_deg,9,0.0823,0.0823,0.1380,0.9673
7,timtrack_betha_super_apo_deg,9,0.0229,0.0500,0.0831,0.9670
8,timtrack_super_pos_y1,9,1.2743,1.2743,1.4441,0.9688
9,timtrack_super_pos_y2,9,0.9920,0.9920,1.0421,0.9674


## Decision

The downstream gate is corrected when both fixes are applied. The independently generated raw Python TimTrack alpha still represents the raw mask/doHough path, not the saved full-sequence MATLAB geofeature alpha stream.


In [8]:
fixed_pass = bool(summary_df.loc[summary_df["variant"] == "alpha_and_klt_fixed", "passes_final_gate"].iloc[0])
raw_alpha_pass = bool(alpha_metrics.loc[alpha_metrics["comparison"] == "raw_python_timtrack_alpha", "passes"].iloc[0])
saved_alpha_pass = bool(alpha_metrics.loc[alpha_metrics["comparison"] == "saved_peak_stream_rebuild", "passes"].iloc[0])
one_step_angle_pass = bool(klt_metrics.loc[klt_metrics["comparison"] == "one_step_handoff_angle_deg", "passes"].iloc[0])
one_step_length_pass = bool(klt_metrics.loc[klt_metrics["comparison"] == "one_step_handoff_length_mm", "passes"].iloc[0])

readiness = pd.DataFrame([
    {
        "gate": "saved full-sequence TimTrack alpha",
        "status": "passes exactly from saved peak stream" if saved_alpha_pass else "fails",
        "ready_for_next": saved_alpha_pass,
    },
    {
        "gate": "independent raw Python alpha",
        "status": "not the saved MATLAB full-sequence alpha stream" if not raw_alpha_pass else "passes",
        "ready_for_next": raw_alpha_pass,
    },
    {
        "gate": "non-compounding KLT handoff",
        "status": "angle and length pass" if one_step_angle_pass and one_step_length_pass else "fails angle or length",
        "ready_for_next": bool(one_step_angle_pass and one_step_length_pass),
    },
    {
        "gate": "corrected alpha + KLT + 2-state Kalman",
        "status": "final FL/PEN/ANG pass" if fixed_pass else "final gate fails",
        "ready_for_next": fixed_pass,
    },
])
readiness_path = OUT / "readiness.csv"
readiness.to_csv(readiness_path, index=False)
display(readiness)
print("Saved:", readiness_path)

summary = {
    "notebook": "50_fix_alpha_klt_drift_pipeline_gate.ipynb",
    "saved_alpha_reconstruction_passes": saved_alpha_pass,
    "raw_python_alpha_passes_saved_full_sequence_target": raw_alpha_pass,
    "one_step_klt_angle_passes": one_step_angle_pass,
    "one_step_klt_length_passes": one_step_length_pass,
    "corrected_alpha_and_klt_final_gate_passes": fixed_pass,
    "raw_python_alpha_rmse_deg": float(alpha_metrics.loc[alpha_metrics["comparison"] == "raw_python_timtrack_alpha", "rmse"].iloc[0]),
    "one_step_klt_angle_rmse_deg": float(klt_metrics.loc[klt_metrics["comparison"] == "one_step_handoff_angle_deg", "rmse"].iloc[0]),
    "one_step_klt_length_rmse_mm": float(klt_metrics.loc[klt_metrics["comparison"] == "one_step_handoff_length_mm", "rmse"].iloc[0]),
    "corrected_FL_rmse_mm": float(summary_df.loc[summary_df["variant"] == "alpha_and_klt_fixed", "FL_rmse_mm"].iloc[0]),
    "corrected_PEN_rmse_deg": float(summary_df.loc[summary_df["variant"] == "alpha_and_klt_fixed", "PEN_rmse_deg"].iloc[0]),
    "corrected_ANG_rmse_deg": float(summary_df.loc[summary_df["variant"] == "alpha_and_klt_fixed", "ANG_rmse_deg"].iloc[0]),
    "remaining_non_oracle_work": [
        "reproduce MATLAB saved full-sequence peak stream from image masks without reading saved geofeatures",
        "port aponeurosis state estimator for fully independent final output",
    ],
}
summary_path = OUT / "summary.json"
summary_path.write_text(json.dumps(summary, indent=2))
print("Saved:", summary_path)
print(json.dumps(summary, indent=2))
display(Markdown("**Decision:** corrected alpha + non-compounding KLT handoff passes the 2-state final gate."))


,gate,status,ready_for_next
0,saved full-sequence TimTrack alpha,passes exactly from saved peak stream,True
1,independent raw Python alpha,not the saved MATLAB full-sequence alpha stream,False
2,non-compounding KLT handoff,angle and length pass,True
3,corrected alpha + KLT + 2-state Kalman,final FL/PEN/ANG pass,True


Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook50_alpha_klt_drift_fix/readiness.csv
Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook50_alpha_klt_drift_fix/summary.json
{
  "notebook": "50_fix_alpha_klt_drift_pipeline_gate.ipynb",
  "saved_alpha_reconstruction_passes": true,
  "raw_python_alpha_passes_saved_full_sequence_target": false,
  "one_step_klt_angle_passes": true,
  "one_step_klt_length_passes": true,
  "corrected_alpha_and_klt_final_gate_passes": true,
  "raw_python_alpha_rmse_deg": 6.201488399133804,
  "one_step_klt_angle_rmse_deg": 0.3013109932225762,
  "one_step_klt_length_rmse_mm": 0.46622294045826423,
  "corrected_FL_rmse_mm": 0.7529551324064312,
  "corrected_PEN_rmse_deg": 0.409723832187515,
  "corrected_ANG_rmse_deg": 0.40972383218751485,
  "remaining_non_oracle_work": [
    "reproduce MATLAB saved full-sequence peak stream from image masks without reading saved geofeatures",
    "port aponeurosis state estimator for fully independent fina

**Decision:** corrected alpha + non-compounding KLT handoff passes the 2-state final gate.